# PyTorch from the Inside: Tensors, Autograd and Precision

The runnable code of [PyTorch from the Inside: Tensors, Autograd and Precision](https://residual-stream.lvtan71.workers.dev/posts/pytorch-from-the-inside/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/pytorch-from-the-inside.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 1 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "pytorch-from-the-inside"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in ['shared/initialization-and-optimizers/optim.py', 'shared/softmax-and-cross-entropy/router4.py', 'shared/features-overfitting-regularization/router_data.py', 'shared/softmax-and-cross-entropy/softmax_basics.py', 'shared/pytorch-from-the-inside/tiny_autograd.py', 'data/router-questions.csv']:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## A tensor is a view of memory

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/pytorch-from-the-inside/#a-tensor-is-a-view-of-memory)

In [ ]:
import torch

x = torch.arange(12.0).reshape(3, 4)
print(f"x: shape {tuple(x.shape)}, strides {x.stride()}, contiguous {x.is_contiguous()}")
t = x.T                                            # a transpose is a view: new strides, same memory
print(f"x.T: shape {tuple(t.shape)}, strides {t.stride()}, contiguous {t.is_contiguous()}, "
      f"same memory {t.data_ptr() == x.data_ptr()}")
row, col = x[1], x[:, 1]                           # slicing is a view too, with an offset
print(f"x[1]: offset {row.storage_offset()}, strides {row.stride()};  "
      f"x[:, 1]: offset {col.storage_offset()}, strides {col.stride()}")
t[0, 1] = 100.0                                    # writing through a view writes the original
print(f"after t[0, 1] = 100: x[1, 0] = {x[1, 0].item()}")
c = t.contiguous()                                 # a real copy, laid out row by row
print(f"t.contiguous(): strides {c.stride()}, same memory {c.data_ptr() == x.data_ptr()}")
try:
    t.view(12)
except RuntimeError as e:
    print(f"t.view(12) fails: {str(e).split('.')[0]}")
print(f"t.reshape(12) works (it copies when it must): {t.reshape(12)[:4].tolist()}")

The post printed:

```text
x: shape (3, 4), strides (4, 1), contiguous True
x.T: shape (4, 3), strides (1, 4), contiguous False, same memory True
x[1]: offset 4, strides (1,);  x[:, 1]: offset 1, strides (4,)
after t[0, 1] = 100: x[1, 0] = 100.0
t.contiguous(): strides (3, 1), same memory False
t.view(12) fails: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces)
t.reshape(12) works (it copies when it must): [0.0, 100.0, 8.0, 1.0]
```

## Autograd is a tape

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/pytorch-from-the-inside/#autograd-is-a-tape)

In [ ]:
import numpy as np

class Tensor:
    """A NumPy array that remembers how it was made, so it can run backward."""
    def __init__(self, data, parents=(), backward=None):
        self.data, self.grad = np.asarray(data, dtype=float), None
        self.parents, self._backward = parents, backward   # the tape: one node per operation

    def __add__(self, other):
        out = Tensor(self.data + other.data, (self, other))
        def backward(g):                                    # sum over broadcast dimensions
            return unbroadcast(g, self.data.shape), unbroadcast(g, other.data.shape)
        out._backward = backward
        return out

    def __matmul__(self, other):
        out = Tensor(self.data @ other.data, (self, other))
        out._backward = lambda g: (g @ other.data.T, self.data.T @ g)   # "Backpropagation, by Hand"'s rules
        return out

    def relu(self):
        out = Tensor(np.maximum(self.data, 0), (self,))
        out._backward = lambda g: (g * (self.data > 0),)
        return out

    def bce_with_logits(self, y):
        """Mean log-loss of sigmoid(self) against labels y, as one operation.
        (Not stable for very large logits; PyTorch's uses log-sum-exp, as in "Cross-Entropy and the Softmax Gradient".)"""
        p = 1 / (1 + np.exp(-self.data))
        out = Tensor(-(y * np.log(p) + (1 - y) * np.log(1 - p)).mean(), (self,))
        out._backward = lambda g: (g * (p - y) / y.size,)             # p - y, "Learning Is Optimization"
        return out

    def backward(self):
        order, seen = [], set()
        def visit(t):                                        # topological order of the tape
            if id(t) not in seen:
                seen.add(id(t))
                for p in t.parents:
                    visit(p)
                order.append(t)
        visit(self)
        self.grad = np.ones_like(self.data)
        for t in reversed(order):                            # from the loss back to the inputs
            if t._backward:
                for p, g in zip(t.parents, t._backward(t.grad)):
                    p.grad = g if p.grad is None else p.grad + g   # a tensor used twice sums its gradients

def unbroadcast(g, shape):
    while g.ndim > len(shape):                               # dimensions added in front
        g = g.sum(0)
    for i, s in enumerate(shape):                            # dimensions stretched from size 1
        if s == 1:
            g = g.sum(i, keepdims=True)
    return g

In [ ]:
import numpy as np
import torch
from tiny_autograd import Tensor

# "Backpropagation, by Hand"'s worked example: one XOR input through the 2-2-1 network.
x, y = np.array([[1.0, 0.0]]), np.array([[1.0]])
W1, b1 = np.array([[0.5, -0.4], [0.3, 0.8]]), np.array([0.0, 0.1])
w2, b2 = np.array([[0.7], [-0.6]]), np.array([0.1])

ours = [Tensor(a) for a in (W1, b1, w2, b2)]
loss = ((Tensor(x) @ ours[0] + ours[1]).relu() @ ours[2] + ours[3]).bce_with_logits(y)
loss.backward()

theirs = [torch.tensor(a, requires_grad=True) for a in (W1, b1, w2, b2)]
h = torch.relu(torch.tensor(x) @ theirs[0] + theirs[1])
ref = torch.nn.functional.binary_cross_entropy_with_logits(h @ theirs[2] + theirs[3], torch.tensor(y))
ref.backward()

print(f"loss: ours {loss.data:.4f}, PyTorch {ref.item():.4f}")
for name, a, b in zip(["W1", "b1", "w2", "b2"], ours, theirs):
    print(f"dL/d{name}: {np.round(a.grad, 4).ravel().tolist()}   largest gap to PyTorch "
          f"{np.abs(a.grad - b.grad.numpy()).max():.1e}")

# Beyond the worked example: random inputs, a weight used twice, a (1, n) bias.
worst = 0.0
for seed in range(200):
    rng = np.random.default_rng(seed)
    X, Y = rng.normal(size=(8, 5)), rng.integers(0, 2, (8, 1)).astype(float)
    W, b = 0.5 * rng.normal(size=(5, 5)), rng.normal(size=(1, 5))
    wo = 0.5 * rng.normal(size=(5, 1))
    ours = [Tensor(a) for a in (W, b, wo)]
    h = ((Tensor(X) @ ours[0] + ours[1]).relu() @ ours[0]).relu()       # W used twice
    (h @ ours[2]).bce_with_logits(Y).backward()
    theirs = [torch.tensor(a, requires_grad=True) for a in (W, b, wo)]
    h = torch.relu(torch.relu(torch.tensor(X) @ theirs[0] + theirs[1]) @ theirs[0])
    torch.nn.functional.binary_cross_entropy_with_logits(h @ theirs[2], torch.tensor(Y)).backward()
    worst = max(worst, max(np.abs(a.grad - t.grad.numpy()).max() for a, t in zip(ours, theirs)))
print(f"200 random cases, shared weight and (1, 5) bias: largest gap to PyTorch {worst:.1e}")

The post printed:

```text
loss: ours 0.4932, PyTorch 0.4932
dL/dW1: [-0.2726, 0.0, 0.0, 0.0]   largest gap to PyTorch 0.0e+00
dL/db1: [-0.2726, 0.0]   largest gap to PyTorch 0.0e+00
dL/dw2: [-0.1947, 0.0]   largest gap to PyTorch 0.0e+00
dL/db2: [-0.3894]   largest gap to PyTorch 0.0e+00
200 random cases, shared weight and (1, 5) bias: largest gap to PyTorch 1.1e-16
```

## The graph PyTorch records

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/pytorch-from-the-inside/#the-graph-pytorch-records)

In [ ]:
import torch

x = torch.tensor([[1.0, 0.0]])
W1 = torch.tensor([[0.5, -0.4], [0.3, 0.8]], requires_grad=True)
w2 = torch.tensor([[0.7], [-0.6]], requires_grad=True)
z = torch.relu(x @ W1) @ w2

# The graph PyTorch recorded while computing z, walked from the output back.
def walk(fn, depth=0):
    if fn is None:
        return
    print("  " * depth + type(fn).__name__)
    for nxt, _ in fn.next_functions:
        walk(nxt, depth + 1)
walk(z.grad_fn)

# The graph is rebuilt on every forward pass, so ordinary Python control flow
# decides its shape: here, how many times a layer is applied.
def count(fn, seen=None):
    seen = set() if seen is None else seen
    if fn is None or fn in seen:
        return 0
    seen.add(fn)
    return 1 + sum(count(nxt, seen) for nxt, _ in fn.next_functions)

W = torch.eye(2, requires_grad=True)
for repeats in (1, 3):
    h = x
    for _ in range(repeats):
        h = torch.relu(h @ W)
    print(f"layer applied {repeats}x: {count(h.grad_fn)} nodes in the graph")

# Under no_grad nothing is recorded: no graph, and no saved activations.
with torch.no_grad():
    print(f"under no_grad: grad_fn = {torch.relu(x @ W1).grad_fn}")

The post printed:

```text
MmBackward0
  ReluBackward0
    MmBackward0
      AccumulateGrad
  AccumulateGrad
layer applied 1x: 3 nodes in the graph
layer applied 3x: 7 nodes in the graph
under no_grad: grad_fn = None
```

## Devices, and why GPU timings lie

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/pytorch-from-the-inside/#devices-and-why-gpu-timings-lie)

In [ ]:
import time
import torch
GPU = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"  # notebook: the post ran on "mps"

def timed(fn, device, trials=10):
    """Median seconds per call. GPU work is queued, not finished, when a call returns,
    so wait for the device before reading the clock."""
    sync = torch.mps.synchronize if device == GPU else (lambda: None)
    fn(); sync()                                        # warm up
    times = []
    for _ in range(trials):
        t0 = time.perf_counter(); fn(); sync()
        times.append(time.perf_counter() - t0)
    return sorted(times)[trials // 2]

cases = [("cpu", torch.float32), (GPU, torch.float32), (GPU, torch.float16), (GPU, torch.bfloat16)]
for n in (256, 1024, 4096):
    row = [f"n = {n:4d}:"]
    for device, dt in cases:
        a = torch.randn(n, n, device=device, dtype=dt)
        b = torch.randn(n, n, device=device, dtype=dt)
        s = timed(lambda: a @ b, device)
        row.append(f"{device} {str(dt).split('.')[1]:8s} {2 * n ** 3 / s / 1e12:5.2f}")
    print("  ".join(row) + "  TFLOP/s", flush=True)

# The same GPU matmul, timed without waiting for it: the clock only sees the launch.
a = torch.randn(4096, 4096, device=GPU)
a @ a; torch.mps.synchronize()
t0 = time.perf_counter(); a @ a; t_launch = time.perf_counter() - t0
torch.mps.synchronize()
print(f"4096 matmul on mps, no synchronize: {t_launch * 1e3:.2f} ms 'measured', "
      f"{timed(lambda: a @ a, 'mps') * 1e3:.2f} ms real")

# Synchronizing after every call also measures the round trip to the GPU.
# Queue many calls, synchronize once, and the GPU can overlap them.
a = torch.randn(256, 256, device=GPU)
a @ a; torch.mps.synchronize()
t0 = time.perf_counter()
for _ in range(200):
    a @ a
torch.mps.synchronize()
queued = (time.perf_counter() - t0) / 200
print(f"256 matmul on mps: {timed(lambda: a @ a, 'mps') * 1e6:.0f} us per call waiting each time, "
      f"{queued * 1e6:.0f} us per call when 200 are queued ({2 * 256 ** 3 / queued / 1e12:.2f} TFLOP/s)")

The post printed:

```text
n =  256:  cpu float32   1.36  mps float32   0.12  mps float16   0.14  mps bfloat16  0.13  TFLOP/s
n = 1024:  cpu float32   1.88  mps float32   1.52  mps float16   2.38  mps bfloat16  2.46  TFLOP/s
n = 4096:  cpu float32   1.45  mps float32   2.99  mps float16   3.43  mps bfloat16  3.42  TFLOP/s
4096 matmul on mps, no synchronize: 0.05 ms 'measured', 47.58 ms real
256 matmul on mps: 230 us per call waiting each time, 30 us per call when 200 are queued (1.12 TFLOP/s)
```

## Numbers, bit by bit

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/pytorch-from-the-inside/#numbers-bit-by-bit)

In [ ]:
import torch

bits = {torch.float32: (8, 23), torch.float16: (5, 10), torch.bfloat16: (8, 7)}
print(f"{'':9s} {'exponent':>8s} {'mantissa':>8s} {'largest':>10s} {'smallest normal':>16s} "
      f"{'smallest subnormal':>19s} {'step at 1.0':>12s}")
for dt, (e, m) in bits.items():
    f = torch.finfo(dt)                                   # a subnormal is the normal minimum's last bits
    print(f"{str(dt).split('.')[1]:9s} {e:8d} {m:8d} {f.max:10.3g} {f.tiny:16.3g} "
          f"{f.tiny * f.eps:19.3g} {f.eps:12.3g}")

show = lambda v, dt: torch.tensor(v, dtype=torch.float32).to(dt).item()
print()
for v in (1e-8, 70_000.0, 1.001):
    print(f"{v:>9g} stored as  float16: {show(v, torch.float16):<12.6g} bfloat16: {show(v, torch.bfloat16):.6g}")

# Adding many small updates to one number, as an optimizer does to a weight.
for dt in (torch.float32, torch.bfloat16):
    w = torch.tensor(1.0, dtype=dt)
    for _ in range(10_000):
        w = w + torch.tensor(1e-4, dtype=dt)
    print(f"1 + 10,000 steps of 0.0001 in {str(dt).split('.')[1]:8s} = {w.item():.4f} (should be 2.0000)")

# FP16's narrow range: small gradients vanish unless the loss is scaled up first.
g = torch.tensor([1e-8, 2e-7, 1e-4])
fmt = lambda t: "[" + ", ".join(f"{v:.3g}" for v in t.tolist()) + "]"
print(f"gradients {fmt(g)} in float16: {fmt(g.half().float())}")
print(f"scaled by 1024, stored, unscaled: {fmt((g * 1024).half().float() / 1024)}")

The post printed:

```text
          exponent mantissa    largest  smallest normal  smallest subnormal  step at 1.0
float32          8       23    3.4e+38         1.18e-38             1.4e-45     1.19e-07
float16          5       10   6.55e+04          6.1e-05            5.96e-08     0.000977
bfloat16         8        7   3.39e+38         1.18e-38            9.18e-41      0.00781

    1e-08 stored as  float16: 0            bfloat16: 1.00117e-08
    70000 stored as  float16: inf          bfloat16: 70144
    1.001 stored as  float16: 1.00098      bfloat16: 1
1 + 10,000 steps of 0.0001 in float32  = 2.0002 (should be 2.0000)
1 + 10,000 steps of 0.0001 in bfloat16 = 1.0000 (should be 2.0000)
gradients [1e-08, 2e-07, 0.0001] in float16: [0, 1.79e-07, 0.0001]
scaled by 1024, stored, unscaled: [1e-08, 2e-07, 0.0001]
```

## The router, in PyTorch

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/pytorch-from-the-inside/#the-router-in-pytorch)

In [ ]:
import numpy as np
import torch
from router_data import split
from router4 import four_way
from softmax_basics import softmax
from optim import Adam

((Xtr, ytr), (Xva, yva), _), _ = four_way(*split(0))
X, y = torch.tensor(Xtr, dtype=torch.float64), torch.tensor(ytr)

# The four-way router as a PyTorch module, trained the way "Initialization and Optimizers" trained it.
model = torch.nn.Linear(X.shape[1], 4, dtype=torch.float64)
torch.nn.init.zeros_(model.weight); torch.nn.init.zeros_(model.bias)
opt = torch.optim.AdamW([{"params": [model.weight], "weight_decay": 0.3},
                         {"params": [model.bias], "weight_decay": 0.0}],   # no decay on biases
                        lr=0.01)

# The same run with "Initialization and Optimizers"'s hand-written AdamW, on the same mini-batches.
W, b, ours = np.zeros((Xtr.shape[1], 4)), np.zeros(4), Adam(0.01, decay=0.3)

rng = np.random.default_rng(0)
for step in range(1, 3001):
    i = rng.choice(len(ytr), 16, replace=False)
    loss = torch.nn.functional.cross_entropy(model(X[i]), y[i])   # takes logits ("Cross-Entropy and the Softmax Gradient")
    opt.zero_grad()                                               # gradients accumulate otherwise
    loss.backward()
    opt.step()
    G = (softmax(Xtr[i] @ W + b) - np.eye(4)[ytr[i]]) / 16
    ours.step(W, Xtr[i].T @ G, "W"); ours.step(b, G.sum(0), "b")
    if step in (1, 10, 3000):
        gap = np.abs(model.weight.detach().numpy().T - W).max()
        print(f"step {step:4d}: loss {loss.item():.4f}, largest gap between PyTorch and optim.py weights {gap:.1e}")

with torch.no_grad():
    acc = (model(torch.tensor(Xva)).argmax(1).numpy() == yva).mean()
print(f"validation accuracy {acc:.3f}")

The post printed:

```text
step    1: loss 1.3863, largest gap between PyTorch and optim.py weights 1.7e-18
step   10: loss 1.1757, largest gap between PyTorch and optim.py weights 2.8e-17
step 3000: loss 0.0428, largest gap between PyTorch and optim.py weights 1.1e-15
validation accuracy 0.750
```

## What a training step keeps in memory

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/pytorch-from-the-inside/#what-a-training-step-keeps-in-memory)

In [ ]:
import torch

# Count the bytes a training step actually keeps, for a small MLP trained
# with AdamW, the way mixed precision usually runs in PyTorch: FP32 weights,
# BF16 math inside autocast.
model = torch.nn.Sequential(torch.nn.Linear(1024, 4096), torch.nn.ReLU(), torch.nn.Linear(4096, 1024))
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
x = torch.randn(64, 1024)
saved = []                                              # what autograd keeps for backward
weights = {p.shape for p in model.parameters() if p.dim() == 2}
with torch.autograd.graph.saved_tensors_hooks(lambda t: saved.append(t) or t, lambda t: t):
    with torch.autocast("cpu", dtype=torch.bfloat16):
        loss = model(x).float().pow(2).mean()
loss.backward()
opt.step()

size = lambda ts: sum(t.numel() * t.element_size() for t in ts)
dtype = lambda ts: str(next(iter(ts)).dtype).split(".")[1]
n = sum(p.numel() for p in model.parameters())
state = [t for s in opt.state.values() for t in s.values() if t.dim() > 0]
parts = {"weights": list(model.parameters()),
         "gradients": [p.grad for p in model.parameters()],
         "Adam m and v": state}
for name, ts in parts.items():
    print(f"{name:13s} {size(ts) / n:4.1f} bytes/param  ({dtype(ts)})")
print(f"{'total':13s} {sum(size(ts) for ts in parts.values()) / n:4.1f} bytes/param for {n:,} parameters")
copies = [t for t in saved if t.dtype == torch.bfloat16 and t.shape in weights]
print(f"during the step: {len(copies)} BF16 weight copies saved for backward, "
      f"{size(copies) / n:.1f} more bytes/param")

The post printed:

```text
weights        4.0 bytes/param  (float32)
gradients      4.0 bytes/param  (float32)
Adam m and v   8.0 bytes/param  (float32)
total         16.0 bytes/param for 8,393,728 parameters
during the step: 1 BF16 weight copies saved for backward, 1.0 more bytes/param
```

## Graphs, eager, and compilation

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/pytorch-from-the-inside/#graphs-eager-and-compilation)

In [ ]:
import time
import torch

def gelu_scale_shift(x, g, b):
    """Four elementwise steps. Eager PyTorch runs each as its own pass over memory."""
    return torch.nn.functional.gelu(x) * g + b + x

def scale_shift(x, g, b):
    """The same without GELU: three steps, almost no arithmetic."""
    return x * g + b + x

x = torch.randn(4096, 4096)
g, b = torch.randn(4096), torch.randn(4096)

def timed(fn, trials=20):
    fn(); times = []
    for _ in range(trials):
        t0 = time.perf_counter(); fn(); times.append(time.perf_counter() - t0)
    return sorted(times)[trials // 2]

for name, fn in [("GELU, scale, shift, add", gelu_scale_shift), ("scale, shift, add", scale_shift)]:
    t0 = time.perf_counter()
    fast = torch.compile(fn)
    fast(x, g, b)
    compile_s = time.perf_counter() - t0
    eager_s, fused_s = timed(lambda: fn(x, g, b)), timed(lambda: fast(x, g, b))
    same = torch.allclose(fast(x, g, b), fn(x, g, b), atol=1e-5)
    print(f"{name:24s} eager {eager_s * 1e3:5.1f} ms   compiled {fused_s * 1e3:4.1f} ms "
          f"({eager_s / fused_s:.1f}x)   first call {compile_s:.1f} s   same result: {same}")
t = timed(lambda: x.clone())
print(f"{'one read + one write':24s} {t * 1e3:5.1f} ms  (x.clone(), the floor for any one-pass kernel)")

The post printed:

```text
GELU, scale, shift, add  eager   9.7 ms   compiled  4.0 ms (2.4x)   first call 1.6 s   same result: True
scale, shift, add        eager   6.7 ms   compiled  1.8 ms (3.7x)   first call 0.0 s   same result: True
one read + one write       1.6 ms  (x.clone(), the floor for any one-pass kernel)
```